In [1]:
import torch
import CIFAR10.config as c
from utils import trainer as t

In [3]:
import os
import hickle as hkl


def dump_network_weights_and_feedback(network, run_name, output_root="weight_dumps"):
    """Save forward weights and fixed feedback matrices for one network."""
    run_dir = os.path.join(output_root, run_name)
    os.makedirs(run_dir, exist_ok=True)

    forward_weights = {}
    feedback_matrices = {}
    for layer_index, layer in enumerate(network.modules()):
        if not hasattr(layer, "weight"):
            continue

        forward_weights[f"layer_{layer_index}"] = layer.weight.detach().cpu().numpy()

        if hasattr(layer, "B"):
            feedback = layer.B
        elif hasattr(layer, "weight_backward"):
            feedback = layer.weight_backward
        else:
            continue
        feedback_matrices[f"layer_{layer_index}"] = feedback.detach().cpu().numpy()

    hkl.dump(forward_weights, os.path.join(run_dir, "forward_weights.hkl"), mode="w")
    hkl.dump(feedback_matrices, os.path.join(run_dir, "feedback_matrices.hkl"), mode="w")
    print(f"Saved weights and feedback matrices to {run_dir}")


In [4]:
# CIFAR-10 Experiment 2: compare four RFA forward initializations with identical B matrices.
activation = torch.nn.Tanh()
hidden_layers = [1024, 512, 256, 128]
'''
feedback_ranges = [0.025, 0.05, 0.1]
random_seeds = [147, 4562, 25823, 769283, 100000]
'''
num_epochs = 50
checkpoint_interval = num_epochs+1

feedback_ranges = [0.05]
random_seeds = [4738]

def rfa_layers(net):
    return [layer for layer in net.modules() if hasattr(layer, "B")]

def match_balanced_weight_norms(balanced_net, xavier_net):
    """Scale each balanced layer to the corresponding Xavier Frobenius norm."""
    with torch.no_grad():
        for balanced_layer, xavier_layer in zip(
            rfa_layers(balanced_net), rfa_layers(xavier_net)
        ):
            balanced_norm = torch.linalg.vector_norm(balanced_layer.weight)
            xavier_norm = torch.linalg.vector_norm(xavier_layer.weight)
            if balanced_norm == 0:
                raise ValueError("Cannot norm-match a zero balanced weight matrix")
            balanced_layer.weight.mul_(xavier_norm / balanced_norm)


def copy_and_check_feedback_matrices(reference_net, networks):
    reference_layers = rfa_layers(reference_net)
    with torch.no_grad():
        for net in networks:
            for reference_layer, layer in zip(reference_layers, rfa_layers(net)):
                layer.B.copy_(reference_layer.B)

    all_equal = True
    for net in networks:
        for layer_index, (reference_layer, layer) in enumerate(
            zip(reference_layers, rfa_layers(net))
        ):
            equal = torch.equal(reference_layer.B, layer.B)
            print(f"B layer {layer_index}: {equal}")
            all_equal = all_equal and equal
    if not all_equal:
        raise RuntimeError("B matrices are not exactly equal; training was not started.")
    print("All B matrices are exactly equal before training.")


for random_seed in random_seeds:
    for feedback_range in feedback_ranges:
        run_prefix = f"rfa_{random_seed}_B{feedback_range:.3f}_2"
        configs = {}
        common_config = dict(
            activation=activation,
            hidden_layers=hidden_layers,
            SEED=random_seed,
            mode="RFA",
            feedback_range=feedback_range,
        )

        configs["balanced"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced",
            run_id=f"{run_prefix}_balanced",
            init_method="arora_balanced",
            arora_std=0.30,
        )
        '''
        configs["balanced_adversary"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced_1000",
            run_id=f"{run_prefix}_balanced_1000",
            init_method="arora_balanced",
            arora_std=1000.0,
        )
        '''
        configs["xavier"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_xavier",
            run_id=f"{run_prefix}_xavier",
            init_method="xavier",
        )
        '''
        configs["balanced_xavier_norm_matched"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced_xavier_norm_matched",
            run_id=f"{run_prefix}_balanced_xavier_norm_matched",
            init_method="arora_balanced",
            arora_std=0.30,
        )
        '''

        '''
        # Match only the dedicated balanced model; configs["balanced"] remains unchanged.
        match_balanced_weight_norms(
            configs["balanced_xavier_norm_matched"]["net"],
            configs["xavier"]["net"],
        )
        '''
        copy_and_check_feedback_matrices(
            configs["balanced"]["net"],
            [config["net"] for config in configs.values()],
        )

        for condition, config in configs.items():
            print(f"\nTraining {condition}: {run_prefix}")
            t.train_network(
                config,
                num_epochs=num_epochs,
                checkpoint_interval=checkpoint_interval,
            )
            dump_network_weights_and_feedback(
                config["net"],
                run_name=config["run_name"],
            )


Initializing layer Linear(in_features=3072, out_features=1024, bias=False) with method 'xavier'
Layer shape: torch.Size([1024, 3072]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=1024, out_features=512, bias=False) with method 'xavier'
Layer shape: torch.Size([512, 1024]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=512, out_features=256, bias=False) with method 'xavier'
Layer shape: torch.Size([256, 512]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=256, out_features=128, bias=False) with method 'xavier'
Layer shape: torch.Size([128, 256]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer Linear(in_features=128, out_features=10, bias=False) with method 'xavier'
Layer shape: torch.Size([10, 128]), nonlinearity: tanh, gain: 1.6666666666666667
All B matrices are exactly equal before training.

Training balanced: rfa_4738_B0.050_2


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Initializing Wandb:


wandb: Currently logged in as: g-aravindadithya (ICLR_2027) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


EPOCH:  1
Time:  3.4641811847686768
EPOCH:  2
Time:  2.998682737350464
EPOCH:  3
Time:  3.1044929027557373
EPOCH:  4
Time:  2.7096176147460938
EPOCH:  5
Time:  3.8199381828308105
EPOCH:  6
Time:  5.376065969467163
EPOCH:  7
Time:  5.359914302825928
EPOCH:  8
Time:  5.1630449295043945
EPOCH:  9
Time:  5.8037803173065186
EPOCH:  10
Time:  5.21257209777832
EPOCH:  11
Time:  4.841360330581665
EPOCH:  12
Time:  4.9645490646362305
EPOCH:  13
Time:  4.769423007965088
EPOCH:  14
Time:  4.67691445350647
EPOCH:  15
Time:  4.926656484603882
EPOCH:  16
Time:  5.050267457962036
EPOCH:  17
Time:  4.767051696777344
EPOCH:  18
Time:  4.546368598937988
EPOCH:  19
Time:  4.740041732788086
EPOCH:  20
Time:  4.713872909545898
EPOCH:  21
Time:  4.73729395866394
EPOCH:  22
Time:  4.694125413894653
EPOCH:  23
Time:  4.559723854064941
EPOCH:  24
Time:  4.6712119579315186
EPOCH:  25
Time:  4.7532758712768555
EPOCH:  26
Time:  4.668154239654541
EPOCH:  27
Time:  5.16987419128418
EPOCH:  28
Time:  4.574558496475

balancedness/B0,▁▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████████
balancedness/B1,▁▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
balancedness/B2,▁▃▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
balancedness/B3,▁███████████████████████████████████████
balancedness/||W0W0T||_F,▁▅▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇███████
balancedness/||W0W0^T - W1^TW1||_F,▁▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████
balancedness/||W1TW1||_F,▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
balancedness/||W1W1T||_F,▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
balancedness/||W1W1^T - W2^TW2||_F,▁▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
balancedness/||W2TW2||_F,▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
+12,...


FINISHED TRAINING :)
Saved weights and feedback matrices to weight_dumps/rfa_4738_B0.050_2_balanced

Training xavier: rfa_4738_B0.050_2
Initializing Wandb:


EPOCH:  1
Time:  7.2790000438690186
EPOCH:  2
Time:  5.787735462188721
EPOCH:  3
Time:  5.831584930419922
EPOCH:  4
Time:  5.234396934509277
EPOCH:  5
Time:  5.801585674285889
EPOCH:  6
Time:  7.221925973892212
EPOCH:  7
Time:  5.996821880340576
EPOCH:  8
Time:  7.001118421554565
EPOCH:  9
Time:  5.8856422901153564
EPOCH:  10
Time:  5.326360702514648
EPOCH:  11
Time:  5.307125806808472
EPOCH:  12
Time:  5.2455153465271
EPOCH:  13
Time:  4.978014230728149
EPOCH:  14
Time:  5.117597579956055
EPOCH:  15
Time:  5.0756988525390625
EPOCH:  16
Time:  5.0609893798828125
EPOCH:  17
Time:  4.939035415649414
EPOCH:  18
Time:  5.1696014404296875
EPOCH:  19
Time:  4.749864339828491
EPOCH:  20
Time:  4.926790952682495
EPOCH:  21
Time:  5.142866373062134
EPOCH:  22
Time:  4.701930046081543
EPOCH:  23
Time:  4.964467763900757
EPOCH:  24
Time:  4.8655407428741455
EPOCH:  25
Time:  4.798722743988037
EPOCH:  26
Time:  4.878748178482056
EPOCH:  27
Time:  4.832614183425903
EPOCH:  28
Time:  6.2712752819061

balancedness/B0,████▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
balancedness/B1,███▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
balancedness/B2,█▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
balancedness/B3,▁████▇▇▇▇▇▆▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
balancedness/||W0W0T||_F,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
balancedness/||W0W0^T - W1^TW1||_F,▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
balancedness/||W1TW1||_F,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇████
balancedness/||W1W1T||_F,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
balancedness/||W1W1^T - W2^TW2||_F,▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
balancedness/||W2TW2||_F,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇████
+12,...


FINISHED TRAINING :)
Saved weights and feedback matrices to weight_dumps/rfa_4738_B0.050_2_xavier


# Experiment 2
RFA feedback-scale sensitivity sweep: Arora-balanced versus Xavier initialization.

Feedback matrices use $B_{ij} \sim \mathrm{Uniform}(-b,b)$ with $b \in \{0.025, 0.05, 0.10\}$. CIFAR-10 runs use 50 epochs.